In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import MaxPool2D,Flatten,Dense,Conv2D,GlobalAveragePooling2D,BatchNormalization,Dropout


2026-03-05 12:11:13.549884: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled")
    except RuntimeError as e:
        print(e)

Memory growth enabled


In [28]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    brightness_range=[0.7,1.3],
    horizontal_flip=True
)
val_datagen=ImageDataGenerator(rescale=1./255)

test_datagen=val_datagen=ImageDataGenerator(rescale=1./255)



train_genrator=train_datagen.flow_from_directory(r"/mnt/c/Users/elwin/OneDrive/Desktop/Project datasets/data/train",
                                                 target_size=(128,128),
                                                batch_size=32,
                                                class_mode='categorical',
                                                shuffle=True)

test_genrator=test_datagen.flow_from_directory(r"/mnt/c/Users/elwin/OneDrive/Desktop/Project datasets/data/test",
                                              target_size=(128,128),
                                              batch_size=32,
                                              class_mode='categorical',
                                              shuffle=False)
val_genrator=val_datagen.flow_from_directory(r"/mnt/c/Users/elwin/OneDrive/Desktop/Project datasets/data/val",
                                              target_size=(128,128),
                                              batch_size=32,
                                              class_mode='categorical',
                                              shuffle=False)

Found 50937 images belonging to 2 classes.
Found 16981 images belonging to 2 classes.
Found 16980 images belonging to 2 classes.


In [29]:

cnn = Sequential([

    Input(shape=(128,128,3)),

    # Block 1
    Conv2D(32,(3,3),activation='relu'),
    BatchNormalization(),
    Conv2D(32,(3,3),activation='relu'),
    MaxPooling2D(),
 

    # Block 2
    Conv2D(64,(3,3),activation='relu'),
    BatchNormalization(),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(),


    # Block 3
    Conv2D(128,(3,3),activation='relu'),
    BatchNormalization(),
    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(),
   


    Flatten(),

    Dense(128,activation='relu'),
    Dense(64,activation='relu'),
    Dropout(0.5),

    Dense(2,activation='sigmoid')
])

In [30]:
cnn.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 124, 124, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 58, 58, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 27, 27, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 27, 27, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 25, 25, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     2,359,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,655,714 (10.13 MB)

 Trainable params: 2,655,266 (10.13 MB)

 Non-trainable params: 448 (1.75 KB)

In [31]:
print(train_genrator.class_indices)

{'awake': 0, 'sleepy': 1}


In [32]:
class_weight = {
    0: 1.0,   
    1: 1.4    
}

In [33]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [34]:
cnn.fit(train_genrator,epochs=12,validation_data=val_genrator,class_weight=class_weight,callbacks=[early_stop])

Epoch 1/12


2026-03-05 13:54:23.770488: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,126,126]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,3,128,128]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-03-05 13:54:23.896406: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,124,124]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,32,126,126]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActiv

1475/1592 ━━━━━━━━━━━━━━━━━━━━ 50s 435ms/step - accuracy: 0.8245 - loss: 0.4995

2026-03-05 14:05:15.597288: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[25,32,126,126]{3,2,1,0}, u8[0]{0}) custom-call(f32[25,3,128,128]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-03-05 14:05:15.733146: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[25,32,124,124]{3,2,1,0}, u8[0]{0}) custom-call(f32[25,32,126,126]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActiv

1592/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step - accuracy: 0.8295 - loss: 0.4864

2026-03-05 14:06:20.939402: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,126,126]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,3,128,128]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-03-05 14:06:21.063679: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,124,124]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,32,126,126]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActiv

1592/1592 ━━━━━━━━━━━━━━━━━━━━ 928s 576ms/step - accuracy: 0.8947 - loss: 0.3154 - val_accuracy: 0.9622 - val_loss: 0.0932
Epoch 2/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 922s 579ms/step - accuracy: 0.9577 - loss: 0.1469 - val_accuracy: 0.9319 - val_loss: 0.1848
Epoch 3/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 733s 460ms/step - accuracy: 0.9670 - loss: 0.1149 - val_accuracy: 0.9816 - val_loss: 0.0505
Epoch 4/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 910s 572ms/step - accuracy: 0.9704 - loss: 0.1024 - val_accuracy: 0.9693 - val_loss: 0.0797
Epoch 5/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 531s 333ms/step - accuracy: 0.9739 - loss: 0.0891 - val_accuracy: 0.9824 - val_loss: 0.0487
Epoch 6/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 527s 331ms/step - accuracy: 0.9757 - loss: 0.0857 - val_accuracy: 0.9280 - val_loss: 0.1727
Epoch 7/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 954s 599ms/step - accuracy: 0.9774 - loss: 0.0802 - val_accuracy: 0.9459 - val_loss: 0.1418
Epoch 8/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 591s 371ms/step - accuracy: 0.9

In [35]:
cnn.evaluate(test_genrator)

530/531 ━━━━━━━━━━━━━━━━━━━━ 0s 365ms/step - accuracy: 0.9822 - loss: 0.0535

2026-03-05 16:18:10.917236: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[21,32,126,126]{3,2,1,0}, u8[0]{0}) custom-call(f32[21,3,128,128]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-03-05 16:18:11.016815: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[21,32,124,124]{3,2,1,0}, u8[0]{0}) custom-call(f32[21,32,126,126]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActiv

531/531 ━━━━━━━━━━━━━━━━━━━━ 196s 369ms/step - accuracy: 0.9848 - loss: 0.0424


[0.042435552924871445, 0.9848065376281738]

In [36]:
val_genrator.reset()

In [37]:
import numpy as np

y_pred_probs = cnn.predict(val_genrator, verbose=1)

531/531 ━━━━━━━━━━━━━━━━━━━━ 86s 160ms/step


In [38]:
y_pred = np.argmax(y_pred_probs, axis=1)

In [39]:
y_true = val_genrator.classes

In [40]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    target_names=list(val_genrator.class_indices.keys())
))

[[8435  156]
 [  95 8294]]
              precision    recall  f1-score   support

       awake       0.99      0.98      0.99      8591
      sleepy       0.98      0.99      0.99      8389

    accuracy                           0.99     16980
   macro avg       0.99      0.99      0.99     16980
weighted avg       0.99      0.99      0.99     16980



In [19]:
y_true = test_genrator.classes
y_pred = np.argmax(cnn.predict(test_genrator), axis=1)

531/531 ━━━━━━━━━━━━━━━━━━━━ 107s 201ms/step


In [21]:
print(len(y_true))
print(len(y_pred))

16981
16981


In [44]:
cnn.save('eye_sleep.h5')